# Step 0 · How to read code without getting lost

**Pain–Comfort Lab · Looking inside a language model** · ⏱️ about 25 minutes · 💻 runs on any computer (no GPU needed)

> ### 🤔 Wonder
> When a line of code runs, **what exactly did it hand back to you?** A number? A list? A table? A block of 1.3 million numbers?

Most people who get lost in machine-learning code are not lost because any single line is hard. They get lost because, three cells later, they no longer know **what kind of thing** a variable is or **what shape** it has. Then every new line feels like magic.

This notebook teaches one habit you will use in every step of the lab:

> ## Run → Look → Explain

1. **Run** a line.
2. **Look** at what came back: its type, its shape, a peek at its values.
3. **Explain** it in plain words before moving on: *"`train_act` is a NumPy array of 32 sentences × 28 layers × 1,536 numbers."*

There is no AI model in this notebook. You'll practise on tiny, made-up data, so every result fits on the screen.

### Five questions to ask about any code cell

| # | Question | Example answer |
|---|---|---|
| 1 | **What goes in?** | a sentence (a string) |
| 2 | **What comes out, and what type is it?** | a PyTorch tensor |
| 3 | **What shape is it, and what does each dimension mean?** | `(1, 9)` → 1 sentence × 9 tokens |
| 4 | **Why this data structure and not another?** | the model only accepts tensors |
| 5 | **Which settings were chosen, and what would change if I changed one?** | `return_tensors='pt'` asks for PyTorch tensors instead of lists |

### Signposts you'll see in every notebook

| Signpost | Meaning |
|---|---|
| 🔧 **Helper cell** | Run it. You can read it, but you don't have to change it. |
| 🧪 **Experiment cell** | Yours to change. Edit sentences or settings and rerun. |
| ✏️ **Predict first** | Write down your guess *before* running. Surprises are where learning happens. |
| 📦 **What came back?** | We stop and inspect the result with `look()`. |
| 🔍 **Code walk-through** | A line-by-line table: what each piece does and *why* it is written that way. |
| 🎛️ **Why this setting?** | The reasoning behind a parameter value. |
| 🧱 **Why this data structure?** | Why a list, array, tensor, table or dictionary was chosen. |
| 🛠️ **Tinker** | Change one thing on purpose and predict what happens (sometimes an error!). |
| 🌱 **Curiosity branch** | A question you could turn into your own investigation (Step 7). |
| 🔭 **Going deeper** | Optional detail for the curious. Safe to skip. |

## Setup

Run the cell below. It makes the lab's small helper library, called **`pclab`**, available. In Colab it downloads the lab from GitHub the first time; on your own computer it uses the copy you downloaded.

`pclab` is not a black box. Every helper in it is explained in a notebook, and you can read any of them at any time with `show_source(...)`.

In [ ]:
# 🔧 Setup: just run this cell. It works in Google Colab and in Jupyter on your own computer.
# It makes the lab's helper library (the `pclab` folder) and the sentence files available.
import subprocess, sys
from pathlib import Path
from importlib.metadata import version
from packaging.version import Version

REPO_URL = 'https://github.com/23menon23/pain-comfort-lab'   # the lab's home on GitHub

if Path('../pclab').exists():                    # you are inside a downloaded copy of the repository
    LAB_ROOT = Path('..').resolve()
else:                                            # Google Colab: download the lab once per session
    LAB_ROOT = Path('pain-comfort-lab').resolve()
    if not LAB_ROOT.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(LAB_ROOT)], check=True)
sys.path.insert(0, str(LAB_ROOT))

if Version(version('transformers')) < Version('4.57'):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=4.57'], check=True)
    print('⚠️ Transformers was updated. Choose Runtime → Restart session, then run this cell again.')
else:
    import pclab
    from pclab import look, show_source
    print('✅ Ready. pclab', pclab.__version__, '| Transformers', version('transformers'))

## Part 1 · The inspection habit: `look()`

`look(thing)` prints a plain-language description of *anything*: what type it is, its shape, the kind of numbers inside, and a small peek. You'll use it constantly.

Here are the five data structures you'll meet in this lab. Run each cell and read the printout.

In [ ]:
# 🧪 A list of sentences, and a dictionary of settings
sentences = ['The tea warmed her throat.', 'The tea burned her throat.']
settings = {'model': 'Qwen/Qwen2.5-1.5B', 'layer': 14, 'seed': 42}

look(sentences, 'sentences')
look(settings, 'settings')

In [ ]:
# 🧪 The same numbers as a NumPy array and as a PyTorch tensor
import numpy as np
import torch

grid = np.array([[0.5, -1.2, 3.0],
                 [2.1, 0.0, -0.7]])
tensor = torch.tensor(grid, dtype=torch.float32)

look(grid, 'grid')
look(tensor, 'tensor')

In [ ]:
# 🧪 A table (pandas DataFrame)
import pandas as pd

table = pd.DataFrame({
    'text': ['The tea warmed her throat.', 'The tea burned her throat.'],
    'category': ['Comfort', 'Pain'],
    'label': [1, 0],
})
look(table, 'table')

### 🧱 Why this data structure? Why so many kinds?

Each structure is good at one job. The lab uses each one where it fits best:

| Structure | Good at | Used in this lab for |
|---|---|---|
| **list** `[...]` | holding anything, in order; easy to grow | lists of sentences, collecting results one by one |
| **dict** `{key: value}` | looking things up by *name* | settings; bundles of named results |
| **NumPy array** | fast math on a grid of numbers that all have the same type and a fixed **shape** | the snapshots we analyse (averages, differences, scores) |
| **PyTorch tensor** | the same as an array, but it can live on a **GPU** and is what the model reads and writes | everything going into or coming out of the model |
| **DataFrame** | tables with named columns that humans read | sentences with their labels; results tables |

**A rule of thumb for this lab:** text lives in lists and DataFrames; numbers *the model* makes are tensors; numbers *we* analyse are NumPy arrays.

Notice that `dtype` changed between `grid` (float64) and `tensor` (float32). **dtype** is the kind of number stored. float32 uses half the memory of float64, and it's the precision this lab runs the model in.

### Converting a tensor to a NumPy array: the "chain"

Inside the model, numbers are tensors, often on the GPU. To analyse them with NumPy we convert them. You'll see this chain in Step 2:

```python
array = tensor.detach().float().cpu().numpy()
```

**✏️ Predict first:** at each link of the chain, does the *shape* change? Does the *type* change? Run the next cell to check.

In [ ]:
# 🔧 Walk the chain one link at a time, looking after each step
step1 = tensor.detach()   # stop tracking it for training (we never train here)
step2 = step1.float()     # make sure the numbers are float32
step3 = step2.cpu()       # move it to main memory (it may have been on the GPU)
step4 = step3.numpy()     # turn it into a NumPy array

for name, value in [('detach()', step1), ('float()', step2), ('cpu()', step3), ('numpy()', step4)]:
    print(f'after .{name:<9} → {pclab.brief(value)}')

**What you should see:** the shape never changes. Only the *kind of container* changes, and only at the last step. Each link exists because NumPy can't read a tensor that is on a GPU or is still being tracked for training.

## Part 2 · Shapes are your map

In Step 2 you'll collect a 3-D block of numbers: **sentences × layers × numbers per token**. Here's a tiny version with 4 sentences, 3 layers and 2 numbers, so you can see all of it.

Picture it as a **stack of 4 small spreadsheets**, one per sentence. Each spreadsheet has 3 rows (layers) and 2 columns (numbers).

In [ ]:
# 🔧 A toy "snapshot block": 4 sentences × 3 layers × 2 numbers
toy = np.arange(24, dtype=float).reshape(4, 3, 2)
look(toy, 'toy', dims=('sentences', 'layers', 'numbers'))
print()
print(toy)

### ✏️ Predict first: what shape comes back?

Write down the shape you expect for each line **before** running the next cell.

| Expression | Your guess | In words |
|---|---|---|
| `toy[0]` | | |
| `toy[:, 1, :]` | | |
| `toy[:, 1, 0]` | | |
| `toy.mean(axis=0)` | | |
| `toy.mean(axis=2)` | | |

In [ ]:
# 🔧 Check your guesses
print('toy[0]           →', toy[0].shape,           '  sentence 0: all its layers and numbers')
print('toy[:, 1, :]     →', toy[:, 1, :].shape,     '  every sentence, layer 1 only')
print('toy[:, 1, 0]     →', toy[:, 1, 0].shape,     '     every sentence, layer 1, number 0')
print('toy.mean(axis=0) →', toy.mean(axis=0).shape, '  averaged over sentences: one "average sentence"')
print('toy.mean(axis=2) →', toy.mean(axis=2).shape, '  averaged over the numbers inside each layer')

### The three shape rules

1. **A single number in a slot removes that dimension.** `toy[0]` picks one sentence, so the sentence dimension disappears: `(3, 2)`.
2. **A colon `:` keeps the whole dimension.** `toy[:, 1, :]` keeps all sentences and all numbers: `(4, 2)`.
3. **`.mean(axis=k)` squashes dimension `k`.** `axis=0` averages across sentences, leaving `(3, 2)`. You'll use exactly this to build an "average comfort sentence" in Step 3.

`axis=-1` means "the last dimension". It's handy because it works whatever the number of dimensions.

### Picking rows with True/False: a boolean mask

To average only the **comfort** sentences, we need to pick some sentences and not others. A **mask** is an array of True/False values, one per sentence.

In [ ]:
# 🔧 Labels for the 4 toy sentences: 1 = Comfort, 0 = Pain
labels = np.array([1, 0, 1, 0])
comfort_mask = labels == 1

look(comfort_mask, 'comfort_mask')
look(toy[comfort_mask], 'toy[comfort_mask]', dims=('sentences', 'layers', 'numbers'))
look(toy[comfort_mask].mean(axis=0), 'average comfort snapshot', dims=('layers', 'numbers'))

### 🔍 Code walk-through

| Code | What it does | Why it's written this way |
|---|---|---|
| `labels == 1` | compares every label to 1 at once | NumPy applies `==` to each item, giving one True/False per sentence |
| `toy[comfort_mask]` | keeps only the sentences where the mask is True | a mask is a quick, readable way to filter by label |
| `.mean(axis=0)` | averages those sentences together | axis 0 is the sentence dimension |

That last line, `activations[labels == 1].mean(axis=0)`, is the **first line of the real experiment** in Step 3. You've already read it.

## Part 3 · The big idea in two dimensions: a direction

Real snapshots have 1,536 numbers. Let's pretend they have only **two**, so we can draw them. The numbers below are **invented**, not from a model.

- 🔴 Red squares: pain sentences. 🔵 Blue circles: comfort sentences.
- The **arrow** goes from the *average pain point* to the *average comfort point*. That arrow is a **direction**.
- The **×** is the **midpoint**, halfway between the two averages.

In [ ]:
# 🔧 Invented 2-number "snapshots"
import matplotlib.pyplot as plt
pclab.use_lab_style()
C = pclab.COLORS

toy_pain = np.array([[1.0, 1.0], [1.4, 1.2], [0.7, 0.8]])
toy_comfort = np.array([[3.0, 2.5], [3.4, 2.8], [2.8, 2.2]])

pain_avg = toy_pain.mean(axis=0)
comfort_avg = toy_comfort.mean(axis=0)
direction = comfort_avg - pain_avg                 # the arrow from pain to comfort
midpoint = (comfort_avg + pain_avg) / 2
unit = direction / np.linalg.norm(direction)       # same arrow, rescaled to length 1

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(*toy_pain.T, color=C['pain'], marker='s', s=60, label='pain (invented)')
ax.scatter(*toy_comfort.T, color=C['comfort'], marker='o', s=60, label='comfort (invented)')
ax.scatter(*midpoint, marker='x', color='black', s=80, label='midpoint')
ax.annotate('', xy=comfort_avg, xytext=pain_avg, arrowprops={'arrowstyle': '->', 'color': C['accent'], 'lw': 2})
ax.set(xlabel='number 1', ylabel='number 2', title='A direction can use several numbers at once')
ax.legend(fontsize=8)
plt.show()

look(direction, 'direction', dims=('numbers',))

The arrow points **diagonally**. It uses *both* numbers. In the real model, no single number out of 1,536 is likely to mean "pain". A concept, if it's there, is probably spread across many numbers, like the overall feel of a song spread across many sliders on a mixing board.

### Scoring: the "shadow" on a ruler

Lay a ruler along the arrow with zero at the midpoint. Each point casts a shadow onto it. Where the shadow lands is the point's **score**: positive means the comfort side, negative means the pain side. The math for a shadow is a **dot product**.

In [ ]:
# 🔧 Score every invented point
points = np.vstack([toy_pain, toy_comfort])        # stack the two groups into one (6, 2) array
scores = (points - midpoint) @ unit                # @ is the dot product

look(points, 'points', dims=('points', 'numbers'))
look(scores, 'scores', dims=('points',))
print('Side of the ruler:', ['comfort' if s > 0 else 'pain' for s in scores])

### 🔍 Code walk-through: tracking shapes through one line

`scores = (points - midpoint) @ unit`

| Piece | Shape | What happens |
|---|---|---|
| `points` | `(6, 2)` | 6 points, 2 numbers each |
| `midpoint` | `(2,)` | one point |
| `points - midpoint` | `(6, 2)` | NumPy subtracts the midpoint from **every row**. This automatic stretching is called **broadcasting**. |
| `@ unit` with `unit` of shape `(2,)` | `(6,)` | each row is dot-multiplied with the arrow, giving one score per point |

**Why subtract the midpoint first?** So that zero on the ruler sits halfway between the groups, and the *sign* of the score tells us the side.

**Why use the length-1 `unit` version of the arrow?** So scores are measured in the same units as the snapshots, whatever the arrow's length.

## Part 4 · Errors are clues, not failures

When shapes don't fit, Python stops with an error. The **last line** of an error message is the most useful one, and shape errors usually *tell you the shapes*.

**✏️ Predict first:** `np.ones((3, 2)) @ np.ones(3)`: will it work?

In [ ]:
# 🛠️ Tinker: cause a shape error on purpose and read it
try:
    np.ones((3, 2)) @ np.ones(3)
except ValueError as error:
    print('ValueError:', error)

# To see the full red error message instead, remove the # on the next line and rerun:
# np.ones((3, 2)) @ np.ones(3)

**How to read it:** `@` needs the *last* dimension of the left side (2) to match the *first* dimension of the right side (3). They don't. When you hit an error like this in a later step:

1. Read the last line of the message.
2. Run `look()` on each variable in the failing line.
3. Write their shapes next to each other and ask which dimensions should match.

## Part 5 · Reading a function someone else wrote

The lab's helpers are short functions. `show_source()` prints any of them with line numbers. Let's read the shortest one: the function that wraps a sentence in the lab's measurement template.

In [ ]:
show_source(pclab.measurement_prompt)

### 🔍 Code walk-through

| Line | What it does | Why |
|---|---|---|
| 1 | `def` starts a function named `measurement_prompt` with three **parameters**. `prefix=...` and `suffix=...` have **default values**. | Defaults are *choices someone made*. You can override them without editing the function, which is how you'll run experiments. |
| 2 | A docstring: a sentence describing the function. | It's what you'd see with `help(pclab.measurement_prompt)`. |
| 3 | Glues prefix + the sentence (with stray spaces removed by `.strip()`) + suffix, and **returns** it. | Returning a value hands the result back to whoever called the function. |

In [ ]:
# 🧪 Call it with the default settings, then override one
prompt = pclab.measurement_prompt('The tea warmed her throat.')
print(prompt)
print(repr(prompt))   # repr() shows hidden characters, such as the line break \n

print(pclab.measurement_prompt('The tea warmed her throat.', suffix='\nIt felt'))

**🎛️ Why `repr()`?** `print()` shows text the way a reader sees it. `repr()` shows it the way Python stores it, including the `\n` line break. When text looks right but behaves oddly, `repr()` usually explains why.

---

## 🧭 Explain it

Answer in your own words (a sentence or two each):

1. What's the difference between a NumPy array and a PyTorch tensor, and why does the lab use both?
2. For an array of shape `(32, 28, 1536)`, what would `array[:, 9, :]` give you, and what shape would it have?
3. In `scores = (points - midpoint) @ unit`, why does `points - midpoint` work even though the shapes are different?
4. In your own words, what is a **direction**, and what is a **score**?

### ✅ Habit checklist for every later step

- [ ] After an important line, I run `look()` on the result.
- [ ] I can say each dimension of a shape in words.
- [ ] When I see a parameter, I ask "what would change if I changed it?"
- [ ] When I get an error, I read the last line first and check shapes.

**Next:** [Step 1 · Meet the model](01_meet_the_model.ipynb)